# Kaiming vs orthogonal h2h initialisation — comparison

Compares models trained with `h2h_init="kaiming"` (Kaiming normal recurrent weights)
against `h2h_init="orthogonal"` (orthogonal scaled by `recurrent_gain=0.9`).

**Important confound:** the kaiming sweep used `init_scale=0.1`; the existing orthogonal
runs in `27_05_26_sweep_cognn_reward_params` used `init_scale=1.0` (pre-dated the
parameter). For a clean apples-to-apples comparison on reward params, use the
`init_scale=0.1` orthogonal runs from `01_06_26_sweep_init_scale` (lick_miss=0,
lick_cost=0 only — those are the only matching reward params available).


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pickle, glob, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as mgs
import pandas as pd
import torch

sys.path.insert(0, str(Path("..").resolve()))
from cxval.models import RNN, ActorCritic
from cxval.envs import TaskEnv
from cxval.agents import Agent
from cxval.analysis import compute_unit_tuning, preferred_value_proportions

KAIMING_DIR  = Path("../results/29_05_26_sweep_kaiming_init")
ORTHOG_DIR   = Path("../results/27_05_26_sweep_cognn_reward_params")   # init_scale=1.0
IS_SWEEP_DIR = Path("../results/01_06_26_sweep_init_scale")            # orthog, init_scale=0.1

LICK_MISS_VALS = [0.0, -0.25, -0.5]
LICK_COST_VALS = [0.0,  0.05,  0.1]
DEVICE         = torch.device("cpu")
ITI_TAIL       = 3
SI_THRESHOLD   = 0.1
SILENT_THR     = 1e-4
STIM_COLORS    = ["#5577aa", "#cc7733", "#44aa55"]

out_dir = KAIMING_DIR / "figures"
out_dir.mkdir(exist_ok=True)
print(f"Output: {out_dir}")

## §0  Load runs

In [ ]:
def _load_dir(sweep_dir, label):
    rows = []
    for _p in sorted(sweep_dir.glob("*/vis_data.pkl")):
        with open(_p, "rb") as _f:
            _vd = pickle.load(_f)
        _psa  = _vd.get("psa_results", {}).get(0, {})
        rows.append(dict(
            label          = label,
            h2h_init       = _vd.get("h2h_init", "orthogonal"),
            init_scale     = float(_vd.get("init_scale", 1.0)),
            reward_lick_miss = float(_vd.get("reward_lick_miss", 0.0)),
            lick_cost      = float(_vd.get("lick_cost", 0.0)),
            seed           = int(_vd.get("seed", 0)),
            spearman_r     = _vd.get("spearman_r", np.nan),
            psa_score      = _psa.get("psa_score",  np.nan),
            lick_high      = _psa.get("high_lick",  np.nan),
            lick_mid       = _psa.get("mid_lick",   np.nan),
            lick_low       = _psa.get("low_lick",   np.nan),
            run_dir        = str(_p.parent),
            run_id         = _vd.get("run_id", _p.parent.name),
            vd_path        = str(_p),
        ))
    return rows

rows_k  = _load_dir(KAIMING_DIR,  "kaiming (is=0.1)")
rows_o  = _load_dir(ORTHOG_DIR,   "orthog (is=1.0)")

# Also load is=0.1 from init_scale sweep — same init_scale as kaiming, lm=lc=0 only
rows_o1 = []
for _p in sorted(IS_SWEEP_DIR.glob("*/vis_data.pkl")):
    with open(_p, "rb") as _f:
        _vd = pickle.load(_f)
    if abs(float(_vd.get("init_scale", 0)) - 0.1) > 1e-6:
        continue
    _psa = _vd.get("psa_results", {}).get(0, {})
    rows_o1.append(dict(
        label="orthog (is=0.1)", h2h_init="orthogonal",
        init_scale=0.1,
        reward_lick_miss=float(_vd.get("reward_lick_miss", 0.0)),
        lick_cost=float(_vd.get("lick_cost", 0.0)),
        seed=int(_vd.get("seed", 0)),
        spearman_r=_vd.get("spearman_r", np.nan),
        psa_score=_psa.get("psa_score", np.nan),
        lick_high=_psa.get("high_lick", np.nan),
        lick_mid=_psa.get("mid_lick",  np.nan),
        lick_low=_psa.get("low_lick",  np.nan),
        run_dir=str(_p.parent), run_id=_vd.get("run_id", _p.parent.name),
        vd_path=str(_p),
    ))

df_k  = pd.DataFrame(rows_k)
df_o  = pd.DataFrame(rows_o)
df_o1 = pd.DataFrame(rows_o1)

print(f"Kaiming runs     : {len(df_k)}  (init_scale=0.1, reward grid)")
print(f"Orthog runs      : {len(df_o)}  (init_scale=1.0, reward grid) — confounded")
print(f"Orthog is=0.1    : {len(df_o1)} (init_scale=0.1, lm=0 lc=0 only)")
for _df, _lbl in [(df_k,"kaiming"),(df_o,"orthog"),(df_o1,"orthog_is01")]:
    if len(_df):
        print(f"  {_lbl}: lick_miss={sorted(_df.reward_lick_miss.unique())}  "
              f"lick_cost={sorted(_df.lick_cost.unique())}  "
              f"seeds={sorted(_df.seed.unique())}")

## §1  Performance across reward parameter grid

Kaiming (left) vs orthogonal init_scale=1.0 (right, confounded — note different init_scale).
For the init_scale-matched comparison see §2.

In [ ]:
# ── Performance heatmaps: kaiming vs orthog (init_scale=1.0) ──────────────
_metrics = [
    ("spearman_r", "Spearman r",   -1,  1, "RdYlGn"),
    ("psa_score",  "PSA score",     0,  1, "RdYlGn"),
    ("lick_high",  "Lick — high",   0,  1, "RdYlGn"),
    ("lick_mid",   "Lick — mid",    0,  1, "RdYlGn"),
    ("lick_low",   "Lick — low",    0,  1, "RdYlGn_r"),
]
_miss_lbls = [str(v) for v in LICK_MISS_VALS]
_cost_lbls = [str(v) for v in LICK_COST_VALS]
_nm, _nc   = len(LICK_MISS_VALS), len(LICK_COST_VALS)

def _grid(df_, col):
    g = np.full((_nc, _nm), np.nan)
    for ci, lc in enumerate(LICK_COST_VALS):
        for mi, lm in enumerate(LICK_MISS_VALS):
            v = df_.loc[
                ((df_.lick_cost - lc).abs() < 1e-6) &
                ((df_.reward_lick_miss - lm).abs() < 1e-6), col
            ].dropna()
            if len(v): g[ci, mi] = v.mean()
    return g

def _hm(ax, data, title, vmin, vmax, cmap):
    im = ax.imshow(data, vmin=vmin, vmax=vmax, cmap=cmap, aspect="auto")
    for ci in range(_nc):
        for mi in range(_nm):
            if np.isfinite(data[ci, mi]):
                ax.text(mi, ci, f"{data[ci,mi]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_xticks(range(_nm)); ax.set_xticklabels(_miss_lbls, fontsize=8)
    ax.set_yticks(range(_nc)); ax.set_yticklabels(_cost_lbls, fontsize=8)
    ax.set_xlabel("lick_miss", fontsize=8); ax.set_ylabel("lick_cost", fontsize=8)
    ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.75, pad=0.02)

fig, axes = plt.subplots(len(_metrics), 2,
                          figsize=(10, 4 * len(_metrics)))
n_seeds_k = df_k["seed"].nunique()
n_seeds_o = df_o["seed"].nunique()

for _ri, (col, title, vmin, vmax, cmap) in enumerate(_metrics):
    _hm(axes[_ri, 0], _grid(df_k, col),
        f"Kaiming (is=0.1)\n{title}  [n={n_seeds_k} seeds]", vmin, vmax, cmap)
    _hm(axes[_ri, 1], _grid(df_o, col),
        f"Orthogonal (is=1.0, confounded)\n{title}  [n={n_seeds_o} seeds]", vmin, vmax, cmap)
    for ax in axes[_ri]:
        ax.spines[["top","right"]].set_visible(False)

fig.suptitle("Kaiming vs orthogonal — reward param grid  (rows=lick_cost, cols=lick_miss)",
             fontsize=11, y=1.01)
plt.tight_layout()
fig.savefig(out_dir / "fig1_perf_grid.png", dpi=150, bbox_inches="tight")
plt.show()

## §2  Lick probability — init_scale-matched comparison (lm=0, lc=0)

Kaiming vs orthogonal, both with `init_scale=0.1`, at the neutral reward params
(lick_miss=0, lick_cost=0).  The orthogonal runs come from the init_scale sweep.

In [ ]:
# ── Matched comparison: kaiming vs orthog (both init_scale=0.1) ───────────
# Filter kaiming to lm=0, lc=0
_df_k0 = df_k[(df_k.reward_lick_miss.abs() < 1e-6) & (df_k.lick_cost.abs() < 1e-6)]
_df_o0 = df_o1  # already filtered to lm=0 lc=0

print(f"Kaiming lm=0 lc=0: {len(_df_k0)} runs, seeds={sorted(_df_k0.seed.unique())}")
print(f"Orthog  lm=0 lc=0: {len(_df_o0)} runs, seeds={sorted(_df_o0.seed.unique())}")

_stim_labels = ["s0 (0%)", "s1 (50%)", "s2 (100%)"]
_lick_cols   = ["lick_low", "lick_mid", "lick_high"]
_ideal_licks = [0.0, 0.5, 1.0]

fig, axes = plt.subplots(1, len(_lick_cols), figsize=(12, 4), sharey=True)
_conditions = [("kaiming (is=0.1)", _df_k0, "#d62728"),
               ("orthog  (is=0.1)", _df_o0, "#1f77b4")]

_xs = np.arange(len(_conditions))

for _ai, (col, lbl, _ideal) in enumerate(zip(_lick_cols, _stim_labels, _ideal_licks)):
    ax = axes[_ai]
    for _xi, (_clbl, _cdf, _col_c) in enumerate(_conditions):
        _vals = _cdf[col].dropna().values
        _m    = _vals.mean() if len(_vals) else np.nan
        _se   = _vals.std() / np.sqrt(len(_vals)) if len(_vals) > 1 else 0.0
        ax.bar(_xi, _m, yerr=_se, capsize=5, color=_col_c, alpha=0.75, width=0.5)
        ax.scatter(np.full(len(_vals), _xi) + np.random.uniform(-0.08, 0.08, len(_vals)),
                   _vals, color=_col_c, s=20, alpha=0.6, zorder=3)
        ax.axhline(_ideal, color="gray", lw=1, ls="--", alpha=0.5)
    ax.set_xticks(_xs)
    ax.set_xticklabels([c for c, *_ in _conditions], fontsize=8, rotation=15, ha="right")
    ax.set_title(lbl, fontsize=10)
    ax.set_ylim(-0.05, 1.05)
    ax.spines[["top","right"]].set_visible(False)
    ax.tick_params(labelsize=8)

axes[0].set_ylabel("Lick probability", fontsize=9)
fig.suptitle("Lick probabilities — kaiming vs orthogonal init  (init_scale=0.1, lm=0, lc=0)\n"
             "dashed = ideal value-matched policy",
             fontsize=11, y=1.02)
plt.tight_layout()
fig.savefig(out_dir / "fig2_lick_matched.png", dpi=150, bbox_inches="tight")
plt.show()

## §3  Trial-aligned heatmaps + weight matrices — representative run

One representative run per init type (lick_miss=0, lick_cost=0, seed=42 where possible).

In [ ]:
# ── Shared helpers (same as init_scale sweep notebook) ───────────────────
def _t(p):
    return np.array(p.detach().cpu().tolist(), dtype=np.float32)

def load_model(vd, state_dict, device=DEVICE):
    obs_dim      = vd["infer_states"].shape[1] + 2 if "infer_states" in vd else None
    if obs_dim is None:
        _sd = state_dict
        obs_dim = _sd["backbone.input2h.weight"].shape[1]
    readout_frac = vd.get("readout_fraction", 1.0)
    H            = vd["hidden_size"]
    bb  = RNN(input_size=obs_dim, hidden_size=H, output_size=1)
    ac  = ActorCritic(backbone=bb, num_actions=2, readout_fraction=readout_frac)
    ac.load_state_dict(state_dict)
    ac.eval()
    return ac.to(device)

def build_snippets(hidden_np, struct, stim_arr, stim_idx, iti_tail):
    stim_len = struct[0]["stim_window"][1]  - struct[0]["stim_window"][0]
    rew_len  = struct[0]["reward_window"][1] - struct[0]["reward_window"][0]
    win_len  = iti_tail + stim_len + rew_len
    snips = []
    for ti, tr in enumerate(struct):
        if stim_arr[ti] != stim_idx: continue
        ss, se = tr["stim_window"]; rs, re = tr["reward_window"]
        iti_s = max(0, ss - iti_tail); pad = iti_tail - (ss - iti_s)
        chunk = np.concatenate([np.zeros((pad, hidden_np.shape[1])),
                                hidden_np[iti_s:se], hidden_np[rs:re]], axis=0)
        snips.append(chunk[:win_len])
    return (np.stack(snips) if snips else None), win_len, stim_len, iti_tail

def stim_mean_act(hidden_np, stim_arr, struct, n_stim):
    acts = []
    for si in range(n_stim):
        chunks = np.concatenate([
            hidden_np[t["stim_window"][0]:t["stim_window"][1]]
            for t, m in zip(struct, stim_arr==si) if m], axis=0)
        acts.append(chunks.mean(axis=0))
    return np.stack(acts, axis=1)

def extract_weights(ac, H):
    n_ro = ac.n_readout; n_act = ac.policy_head.out_features
    W_in = _t(ac.backbone.input2h.weight)
    W_rec = _t(ac.backbone.h2h.weight)
    W_val = np.full((H, 1), np.nan); W_val[:n_ro, 0] = _t(ac.value_head.weight)[0]
    W_pol = np.full((H, n_act), np.nan); W_pol[:n_ro, :] = _t(ac.policy_head.weight).T
    return W_in, W_rec, W_val, W_pol, n_ro

print("Helpers defined.")

In [ ]:
# ── Load representative runs (lm=0, lc=0, seed=42) ───────────────────────
SEED_SHOW = 42

def _find_rep(df_, lm=0.0, lc=0.0, seed=SEED_SHOW):
    sub = df_[(df_.reward_lick_miss.abs() < 1e-6) &
              (df_.lick_cost.abs() < 1e-6) &
              (df_.seed == seed)]
    if len(sub) == 0:
        sub = df_[(df_.reward_lick_miss.abs() < 1e-6) & (df_.lick_cost.abs() < 1e-6)]
        if len(sub) == 0: return None
        seed = int(sub.seed.iloc[0])
    row = sub.iloc[0]
    with open(row.vd_path, "rb") as f: vd = pickle.load(f)
    return vd, Path(row.run_dir)

_rep_k  = _find_rep(df_k)
_rep_o1 = _find_rep(df_o1)

print("Kaiming rep :", _rep_k[1].name  if _rep_k  else "not found")
print("Orthog  rep :", _rep_o1[1].name if _rep_o1 else "not found")

In [ ]:
# ── Compute sort index from kaiming representative ────────────────────────
if _rep_k is None:
    print("No kaiming representative run found — cannot compute sort index")
else:
    _vd_k, _rd_k = _rep_k
    _H       = _vd_k["hidden_size"]
    _n_stim  = _vd_k["n_stimuli"]
    _stimuli = _vd_k["stimuli"]
    _struct_k = _vd_k["infer_trial_structure"]
    _stim_k   = np.array([t["stimulus"] for t in _struct_k])

    _stim_h = _vd_k["infer_activations"]["stim_hidden"]   # (n_trials, ts, H)
    _mean_ctx = np.zeros((_H, _n_stim))
    for _si in range(_n_stim):
        _m = _stim_k == _si
        _mean_ctx[:, _si] = _stim_h[_m].mean(axis=(0,1))

    _max_abs  = _mean_ctx.max(axis=1)
    _si_range = _mean_ctx.max(axis=1) - _mean_ctx.min(axis=1)
    _si_sum   = np.abs(_mean_ctx.sum(axis=1))
    _si_score = np.where(_si_sum > SILENT_THR, _si_range/(_si_sum+1e-12), 0.0)
    _pref     = np.argmax(_mean_ctx, axis=1)
    _sel_mask = (_si_score >= SI_THRESHOLD) & (_max_abs >= SILENT_THR)

    _grp_idx, _grp_sz, _grp_lbl = [], [], []
    for _si in range(_n_stim):
        _g = np.where(_sel_mask & (_pref==_si))[0]
        _g = _g[np.argsort(-_si_score[_g])]
        _grp_idx.append(_g); _grp_sz.append(len(_g)); _grp_lbl.append(_stimuli[_si])
    _ns = np.where(~_sel_mask)[0][np.argsort(-_si_score[np.where(~_sel_mask)[0]])]
    _grp_idx.append(_ns); _grp_sz.append(len(_ns)); _grp_lbl.append("non-sel")
    _SORT_IDX = np.concatenate(_grp_idx).astype(int)
    _boundaries = []
    _cursor = 0
    for _gi, _sz in enumerate(_grp_sz):
        _cursor += _sz
        if _gi < len(_grp_sz) - 1: _boundaries.append(_cursor - 0.5)
    print(f"Sort index: {list(zip(_grp_lbl, _grp_sz))}")

In [ ]:
# ── Side-by-side trial-aligned heatmaps ──────────────────────────────────
if _rep_k is None or _rep_o1 is None:
    print("Need both representative runs to compare")
else:
    _vd_o1, _rd_o1 = _rep_o1
    _reps = [("Kaiming (is=0.1)", _vd_k, _rd_k),
             ("Orthogonal (is=0.1)", _vd_o1, _rd_o1)]

    _stim_len = _struct_k[0]["stim_window"][1] - _struct_k[0]["stim_window"][0]
    _rew_len  = _struct_k[0]["reward_window"][1] - _struct_k[0]["reward_window"][0]
    _win_len  = ITI_TAIL + _stim_len + _rew_len
    _tick_pos = [0, ITI_TAIL, ITI_TAIL+_stim_len, _win_len-1]
    _xlbls    = {0:"ITI", ITI_TAIL:"stim on",
                 ITI_TAIL+_stim_len:"rew on", _win_len-1:"end"}

    # Shared vmax across both models
    _all_vals = []
    for _, _vd_, _ in _reps:
        _hid = np.array(_vd_["infer_activations"]["hidden_states"])
        _sa  = np.array([t["stimulus"] for t in _vd_["infer_trial_structure"]])
        for _si in range(_n_stim):
            _s, *_ = build_snippets(_hid, _vd_["infer_trial_structure"], _sa, _si, ITI_TAIL)
            if _s is not None: _all_vals.append(_s.mean(axis=0))
    _vmax_shared = float(np.nanpercentile(np.concatenate(_all_vals), 99)) if _all_vals else 1.0

    fig, _ax_all = plt.subplots(
        2 * len(_reps), _n_stim + 1,
        figsize=(3.5 * _n_stim + 0.5, 4.5 * len(_reps)),
        gridspec_kw={"height_ratios": [4, 1.2] * len(_reps),
                     "width_ratios": [1.0]*_n_stim + [0.04]},
        squeeze=False,
    )
    for _mi, (_lbl, _vd_, _rd_) in enumerate(_reps):
        _hid  = np.array(_vd_["infer_activations"]["hidden_states"])
        _sa   = np.array([t["stimulus"] for t in _vd_["infer_trial_structure"]])
        _stru = _vd_["infer_trial_structure"]
        _ymax = 0.0; _im = None
        for _si in range(_n_stim):
            _row_h = _mi*2; _row_p = _mi*2+1
            _ax_h = _ax_all[_row_h, _si]; _ax_p = _ax_all[_row_p, _si]
            _s, *_ = build_snippets(_hid, _stru, _sa, _si, ITI_TAIL)
            if _s is None: _ax_h.set_visible(False); _ax_p.set_visible(False); continue
            _ymax = max(_ymax, float(_s.mean(axis=(0,2)).max()))
            _ms = _s.mean(axis=0)[:, _SORT_IDX].T
            _im = _ax_h.imshow(_ms, aspect="auto", cmap="hot",
                               vmin=0, vmax=_vmax_shared, interpolation="nearest")
            for _by in _boundaries:
                _ax_h.axhline(_by, color="white", lw=0.8, ls=":", alpha=0.7)
            _ax_h.axvline(ITI_TAIL-0.5, color="white", lw=1.2, ls="--", alpha=0.6)
            _ax_h.axvline(ITI_TAIL+_stim_len-0.5, color="white", lw=1.2, ls="--", alpha=0.6)
            _ax_h.set_xticks([]); _ax_h.set_xlim(-0.5, _win_len-0.5)
            _ax_h.spines[["top","right"]].set_visible(False)
            if _si == 0: _ax_h.set_ylabel(f"{_lbl}\n(n={_H})", fontsize=8)
            if _mi == 0: _ax_h.set_title(_stimuli[_si], fontsize=9)
            _pop = _s.mean(axis=(0,2))
            _ax_p.plot(range(_win_len), _pop, color=STIM_COLORS[_si % len(STIM_COLORS)], lw=1.5)
            _ax_p.axvline(ITI_TAIL-0.5, color="gray", lw=1, ls="--", alpha=0.5)
            _ax_p.axvline(ITI_TAIL+_stim_len-0.5, color="gray", lw=1, ls="--", alpha=0.5)
            _ax_p.set_ylim(0, _ymax * 1.15)
            _ax_p.spines[["top","right"]].set_visible(False)
            if _mi == len(_reps)-1:
                _ax_p.set_xticks(_tick_pos)
                _ax_p.set_xticklabels([_xlbls[p] for p in _tick_pos], fontsize=7, rotation=35)
            else:
                _ax_p.set_xticks([])
            if _si == 0: _ax_p.set_ylabel("Pop avg act", fontsize=7)
            _ax_p.tick_params(labelsize=7)
        if _im:
            plt.colorbar(_im, cax=_ax_all[_mi*2, _n_stim])
        _ax_all[_mi*2+1, _n_stim].set_visible(False)

    fig.suptitle(f"Trial-aligned heatmaps — kaiming vs orthogonal  [lm=0, lc=0, seed={SEED_SHOW}]",
                 fontsize=11, y=1.01)
    plt.tight_layout()
    fig.savefig(out_dir / "fig3_heatmaps.png", dpi=150, bbox_inches="tight")
    plt.show()

## §4  Weight matrices — kaiming vs orthogonal

In [ ]:
# ── Weight matrices side-by-side ─────────────────────────────────────────
if _rep_k is None or _rep_o1 is None:
    print("Need both representative runs")
else:
    _bundles = []
    for _lbl, _vd_, _rd_ in [("Kaiming (is=0.1)", _vd_k, _rd_k),
                              ("Orthogonal (is=0.1)", _vd_o1, _rd_o1)]:
        _model_path = _rd_ / "model.pt"
        if not _model_path.exists():
            print(f"  {_lbl}: model.pt missing"); continue
        _ac   = load_model(_vd_, torch.load(_model_path, map_location="cpu"))
        _hid  = np.array(_vd_["infer_activations"]["hidden_states"])
        _sa   = np.array([t["stimulus"] for t in _vd_["infer_trial_structure"]])
        _stru = _vd_["infer_trial_structure"]
        _S    = stim_mean_act(_hid, _sa, _stru, _n_stim)
        _Wi, _Wr, _Wv, _Wp, _n_ro = extract_weights(_ac, _H)
        _bundles.append((_lbl, _Wi, _Wr, _Wv, _Wp, _S, _n_ro))

    if len(_bundles) == 2:
        _col_w = [_n_stim, min(_bundles[0][1].shape[1], 10),
                  min(_H, 20), 2, _bundles[0][3].shape[1]*2]
        _ptitles = ["Stim activation", "W_in", "W_rec", "W_val", "W_pol"]

        # Shared colormap limits across both models
        _panel_lims = []
        for _pi, _div in enumerate([False, True, True, True, True]):
            _combined = np.concatenate([
                _bundles[0][_pi+1][np.isfinite(_bundles[0][_pi+1])],
                _bundles[1][_pi+1][np.isfinite(_bundles[1][_pi+1])],
            ])
            if _div:
                _lim = float(np.abs(_combined).max()) if len(_combined) else 1.0
                _panel_lims.append((-_lim, _lim))
            else:
                _panel_lims.append((0.0, float(np.nanpercentile(_combined, 99)) if len(_combined) else 1.0))

        _cmaps = ["hot","RdBu_r","RdBu_r","RdBu_r","RdBu_r"]
        fig = plt.figure(figsize=(sum(_col_w)*18/sum(_col_w)+1.2, 12))
        _outer = mgs.GridSpec(2, 1, figure=fig, hspace=0.35, top=0.90, bottom=0.05)

        for _mi, (_lbl, W_in, W_rec, W_val, W_pol, S_act, _n_ro_) in enumerate(_bundles):
            _inner = mgs.GridSpecFromSubplotSpec(
                2, len(_ptitles)+1, subplot_spec=_outer[_mi],
                hspace=0.08, height_ratios=[4,1.2],
                width_ratios=_col_w+[0.25], wspace=0.18,
            )
            _panels = [S_act[_SORT_IDX,:], W_in[_SORT_IDX,:],
                       W_rec[_SORT_IDX,:][:,_SORT_IDX],
                       W_val[_SORT_IDX,:], W_pol[_SORT_IDX,:]]
            for _pi, (_data, _cmap, (_vmin_,_vmax_), _pt) in enumerate(
                    zip(_panels, _cmaps, _panel_lims, _ptitles)):
                _ax_h = fig.add_subplot(_inner[0,_pi])
                _ax_d = fig.add_subplot(_inner[1,_pi])
                _im   = _ax_h.imshow(_data, aspect="auto", cmap=_cmap,
                                     vmin=_vmin_, vmax=_vmax_, interpolation="nearest")
                if _pi in (3,4) and _n_ro_ < _H:
                    _ax_h.axhline(_n_ro_-0.5, color="cyan", lw=1.2, ls="--", alpha=0.8)
                if _pi == 2:
                    for _by in _boundaries:
                        _ax_h.axhline(_by, color="white", lw=0.7, ls=":", alpha=0.6)
                        _ax_h.axvline(_by, color="white", lw=0.7, ls=":", alpha=0.6)
                _ax_h.set_title(_pt, fontsize=8, pad=2)
                _ax_h.tick_params(left=(_pi==0), labelleft=(_pi==0),
                                  bottom=False, labelbottom=False)
                _ax_h.spines[["top","right","bottom","left"]].set_visible(False)
                if _pi==0:
                    _ax_h.set_yticks([0,_H//2,_H-1]); _ax_h.tick_params(labelsize=7)
                    _ax_h.set_ylabel(f"{_lbl}\n(n={_H})", fontsize=8)
                    _ax_h.set_xticks(range(_n_stim))
                    _ax_h.set_xticklabels([f"s{i}" for i in range(_n_stim)], fontsize=7)
                    _ax_h.xaxis.set_tick_params(bottom=True, labelbottom=True)
                _valid = _data[np.isfinite(_data)]
                if len(_valid):
                    _ax_d.hist(_valid, bins=40,
                               color="steelblue" if _cmap=="hot" else "slategray",
                               alpha=0.75, density=True, linewidth=0)
                    if _cmap=="RdBu_r": _ax_d.axvline(0, color="k", lw=0.8, ls="--", alpha=0.5)
                    _ax_d.set_xlim(_vmin_, _vmax_)
                _ax_d.tick_params(labelsize=7)
                _ax_d.spines[["top","right"]].set_visible(False)
                if _pi!=0: _ax_d.tick_params(labelleft=False)
                else: _ax_d.set_ylabel("density", fontsize=7)
                _ax_d.set_xlabel("act" if _pi==0 else "weight", fontsize=7)
                if _pi == len(_ptitles)-1:
                    _cb = fig.add_subplot(_inner[0,-1])
                    fig.colorbar(_im, cax=_cb); _cb.tick_params(labelsize=6)
                    fig.add_subplot(_inner[1,-1]).set_visible(False)

        fig.suptitle(
            f"Weight matrices — kaiming vs orthogonal  [lm=0, lc=0, seed={SEED_SHOW}]\n"
            "Shared colormap ranges  |  cyan = readout boundary",
            fontsize=10)
        fig.savefig(out_dir/"fig4_weights.png", dpi=150, bbox_inches="tight")
        plt.show()

## §5  Training dynamics — kaiming only

Orthogonal runs in `27_05_26_sweep_cognn_reward_params` predate checkpoint saving
and are not available here. Only kaiming checkpoints are shown.

In [ ]:
# ── Training dynamics for kaiming (lm=0, lc=0), all seeds ────────────────
_period_names = ["ITI", "Stim", "Reward"]
_all_seeds    = sorted(df_k["seed"].unique())

def run_inference_safe(ac, vd):
    if any(torch.isnan(p).any() for p in ac.parameters()): return None
    env_kw = {k: vd[k] for k in
              ("reward_lick","reward_no_lick","reward_lick_miss","lick_cost") if k in vd}
    env = TaskEnv(states=vd["infer_states"],
                  reward_availability=vd["infer_reward_availability"], **env_kw)
    agent = Agent(ac, device=DEVICE); agent.reset()
    obs, _ = env.reset(); hs = []
    try:
        done = False
        while not done:
            action, _, _ = agent.act(obs)
            hs.append(np.array(agent.hidden.detach().squeeze(0).cpu().tolist(), dtype=np.float32))
            obs, _, done, _, _ = env.step(action)
    except (ValueError, RuntimeError): return None
    return np.array(hs)

_seed_dyn  = {}
_ckpt_grid = None

for _seed in _all_seeds:
    _row = df_k[(df_k.reward_lick_miss.abs()<1e-6) & (df_k.lick_cost.abs()<1e-6) &
                (df_k.seed==_seed)]
    if len(_row) == 0: continue
    with open(_row.vd_path.iloc[0], "rb") as f: _vd_ = pickle.load(f)
    _rd_    = KAIMING_DIR / _vd_["run_id"]
    _ckpts  = sorted((_rd_/"checkpoints").glob("checkpoint_*.pt"))
    if not _ckpts: continue

    _struct  = _vd_["infer_trial_structure"]
    _stim_a  = np.array([t["stimulus"] for t in _struct])
    _stim_l  = _struct[0]["stim_window"][1] - _struct[0]["stim_window"][0]
    _rew_l   = _struct[0]["reward_window"][1] - _struct[0]["reward_window"][0]
    _win_l   = ITI_TAIL + _stim_l + _rew_l
    _psl = [slice(0,ITI_TAIL), slice(ITI_TAIL,ITI_TAIL+_stim_l), slice(ITI_TAIL+_stim_l,None)]
    _ckpt_trials = [int(_c.stem.split("_")[1]) for _c in _ckpts]
    if _ckpt_grid is None: _ckpt_grid = _ckpt_trials

    _results = {_si: [[] for _ in _period_names] for _si in range(_vd_["n_stimuli"])}
    print(f"  seed={_seed}: {len(_ckpts)} ckpts...", end=" ", flush=True)
    for _cf in _ckpts:
        _sd_ = torch.load(_cf, map_location="cpu")
        _ac_ = load_model(_vd_, _sd_)
        _hid = run_inference_safe(_ac_, _vd_)
        if _hid is None:
            for _si in range(_vd_["n_stimuli"]):
                for _pi in range(len(_period_names)): _results[_si][_pi].append(np.nan)
            continue
        for _si in range(_vd_["n_stimuli"]):
            _sn, *_ = build_snippets(_hid, _struct, _stim_a, _si, ITI_TAIL)
            if _sn is None:
                for _pi in range(len(_period_names)): _results[_si][_pi].append(np.nan)
                continue
            _mn = _sn.mean(axis=0)
            for _pi, _sl in enumerate(_psl): _results[_si][_pi].append(float(_mn[_sl].mean()))
    print("done")
    _seed_dyn[_seed] = {"results": _results, "n_stim": _vd_["n_stimuli"],
                        "stimuli": _vd_["stimuli"]}

# plot
_ymaxes = [max([v for _d in _seed_dyn.values() for _si in range(_d["n_stim"])
                for v in _d["results"][_si][_pi] if not np.isnan(v)], default=1.0) * 1.15
           for _pi in range(len(_period_names))]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
if _ckpt_grid and _seed_dyn:
    _ns = _seed_dyn[next(iter(_seed_dyn))]["n_stim"]
    _stims = _seed_dyn[next(iter(_seed_dyn))]["stimuli"]
    for _pi, (_pn, ax) in enumerate(zip(_period_names, axes)):
        for _si in range(_ns):
            _vps = [_seed_dyn[s]["results"][_si][_pi] for s in _seed_dyn]
            _arr = np.array(_vps, dtype=float)
            _m   = np.nanmean(_arr, axis=0)
            _se  = np.nanstd(_arr, axis=0) / np.sqrt(np.sum(np.isfinite(_arr), axis=0).clip(1))
            _c   = STIM_COLORS[_si % len(STIM_COLORS)]
            ax.plot(_ckpt_grid, _m, color=_c, lw=1.8, marker="o", ms=3, label=_stims[_si])
            ax.fill_between(_ckpt_grid, _m-_se, _m+_se, color=_c, alpha=0.2, linewidth=0)
        ax.set_ylim(0, _ymaxes[_pi]); ax.set_xlabel("Training trial", fontsize=8)
        ax.set_title(f"{_pn} period", fontsize=9); ax.tick_params(labelsize=7)
        ax.spines[["top","right"]].set_visible(False)
        if _pi == 0:
            ax.set_ylabel("Mean pop activation", fontsize=8)
            ax.legend(fontsize=7, title=f"n={len(_seed_dyn)} seeds", title_fontsize=6)
else:
    for ax in axes: ax.text(0.5,0.5,"no data",ha="center",va="center",transform=ax.transAxes)
fig.suptitle("Training dynamics — kaiming init  [lm=0, lc=0  mean±SEM]",
             fontsize=11, y=1.02)
plt.tight_layout()
fig.savefig(out_dir/"fig5_dynamics_kaiming.png", dpi=150, bbox_inches="tight")
plt.show()